In [2]:
# IMPORT LIBRARIES

import os
import sys
import re
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data handling
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Transformers 
from transformers import DistilBertTokenizer, DistilBertModel


print(" ALL LIBRARIES IMPORTED SUCCESSFULLY!")

print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(" Running on CPU ")


 ALL LIBRARIES IMPORTED SUCCESSFULLY!
PyTorch: 2.5.1
NumPy: 2.0.1
Pandas: 2.3.3
CUDA Available: False
 Running on CPU 


In [3]:
# CONFIGURATION

# DATASET PATH
BASE_DIR = "KIDO/"
IMAGES_DIR = os.path.join(BASE_DIR, "Images", "Emotion")
TEXTS_DIR = os.path.join(BASE_DIR, "Texts", "Emotion")

TRAIN_CSV = os.path.join(TEXTS_DIR, "Emotion_Train.csv")
TEST_CSV = os.path.join(TEXTS_DIR, "Emotion_Test.csv")

# MODEL SAVE DIRECTORY
MODEL_DIR = "Emotion_Models/"
os.makedirs(MODEL_DIR, exist_ok=True)

# TRAINING PARAMETE
BATCH_SIZE = 16
EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_TEXT_LENGTH = 50
EMBEDDING_DIM = 300

# DEVICE
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# PRINT CONFIGURATION
print("KIDO EMOTION CLASSIFICATION - TRAINING CONFIGURATION")

print(f"Images Directory: {IMAGES_DIR}")
print(f"Texts Directory:  {TEXTS_DIR}")
print(f"Train CSV:       {TRAIN_CSV}")
print(f"Test CSV:        {TEST_CSV}")
print(f"Model Save Dir:  {MODEL_DIR}")
print(f"Device:          {DEVICE}")
print(f"Batch Size:      {BATCH_SIZE}")
print(f"Epochs:          {EPOCHS}")
print(f"Patience:        {EARLY_STOPPING_PATIENCE}")


KIDO EMOTION CLASSIFICATION - TRAINING CONFIGURATION
Images Directory: KIDO/Images\Emotion
Texts Directory:  KIDO/Texts\Emotion
Train CSV:       KIDO/Texts\Emotion\Emotion_Train.csv
Test CSV:        KIDO/Texts\Emotion\Emotion_Test.csv
Model Save Dir:  Emotion_Models/
Device:          cpu
Batch Size:      16
Epochs:          15
Patience:        3


In [4]:
# CHECK DATASET

print("CHECKING DATASET")


# Check if directories exist
if os.path.exists(IMAGES_DIR):
    print(f" Images: {IMAGES_DIR}")
    for split in ['train', 'test']:
        for label in ['Happiness', 'Sadness']:
            path = os.path.join(IMAGES_DIR, split, label)
            if os.path.exists(path):
                count = len([f for f in os.listdir(path) if f.endswith(('.png', '.jpg', '.jpeg'))])
                print(f"   {split}/{label}: {count} images")
            else:
                print(f"    {split}/{label} not found")
else:
    print(f" Images NOT found: {IMAGES_DIR}")
    print("   Please check your path and folder structure.")

if os.path.exists(TEXTS_DIR):
    print(f" Texts: {TEXTS_DIR}")
    if os.path.exists(TRAIN_CSV):
        print(f"    Emotion_Train.csv")
        df = pd.read_csv(TRAIN_CSV, header=None, nrows=2)
        print(f"      Preview:\n{df.iloc[:, :4]}")
    else:
        print(f"    Emotion_Train.csv NOT found")
    if os.path.exists(TEST_CSV):
        print(f"    Emotion_Test.csv")
        df = pd.read_csv(TEST_CSV, header=None, nrows=2)
        print(f"      Preview:\n{df.iloc[:, :4]}")
    else:
        print(f"    Emotion_Test.csv NOT found")
else:
    print(f" Texts NOT found: {TEXTS_DIR}")



CHECKING DATASET
 Images: KIDO/Images\Emotion
   train/Happiness: 4614 images
   train/Sadness: 4614 images
   test/Happiness: 816 images
   test/Sadness: 816 images
 Texts: KIDO/Texts\Emotion
    Emotion_Train.csv
      Preview:
                0                                                  1  \
0  103-4L-974-M-S  Ormanlarda yangın oluyor ama biz sadece izleye...   
1  207-6C-543-M-S            Bir insanın ölümünü gördüğümde üzülürüm   

                                                   2        3  
0  There is fire in the forests. But we can only ...  Sadness  
1                  I feel sad when I see someone die  Sadness  
    Emotion_Test.csv
      Preview:
                0                                                  1  \
0  101-1H-621-M-S  Ali evde çok sıkıldı Cem teneffüse çıkmadığı i...   
1  108-3A-362-F-S                Doğaya zarar vermek beni çok üzüyor   

                                                   2        3  
0  Ali was bored at home and Cem was very up

In [5]:
#  DATA LOADING FUNCTIONS

def preprocess_text(text):
    """Clean and preprocess text."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_kido_data():
    """
    Load KIDO dataset from local structure.
    
    CSV FORMAT: NO HEADERS!
    - Column 0: ID (e.g., "103-4L-974-M-S")
    - Column 2: Text (self-reflection)
    - Column 3: Label (Happiness/Sadness)
    """
    
    data = []
    
    # Check if CSV files exist
    if not os.path.exists(TRAIN_CSV):
        print(f" Train CSV not found: {TRAIN_CSV}")
        return create_dummy_data()
    
    if not os.path.exists(TEST_CSV):
        print(f" Test CSV not found: {TEST_CSV}")
        return create_dummy_data()
    
    # Load CSV files with NO HEADER
    print(" Loading CSV files...")
    train_df = pd.read_csv(TRAIN_CSV, header=None)
    test_df = pd.read_csv(TEST_CSV, header=None)
    
    print(f"   Train CSV: {len(train_df)} rows")
    print(f"   Test CSV: {len(test_df)} rows")
    
    # Column indices (0-based): 0=ID, 2=Text, 3=Label
    ID_COL, TEXT_COL, LABEL_COL = 0, 2, 3
    
    # Process training data
    print("\n Processing training data...")
    train_success = 0
    
    for _, row in train_df.iterrows():
        try:
            img_id = str(row[ID_COL]).strip()
            text = str(row[TEXT_COL]) if pd.notna(row[TEXT_COL]) else ""
            label = str(row[LABEL_COL]).strip() if pd.notna(row[LABEL_COL]) else ""
            
            # Map label
            if label.lower() in ['happiness', 'happy']:
                label = 'Happiness'
            elif label.lower() in ['sadness', 'sad']:
                label = 'Sadness'
            else:
                continue
            
            # Check if image exists
            img_path = os.path.join(IMAGES_DIR, "train", label, f"{img_id}.png")
            if not os.path.exists(img_path):
                img_path = os.path.join(IMAGES_DIR, "train", label, f"{img_id}.jpg")
                if not os.path.exists(img_path):
                    continue
            
            data.append({
                'image_id': img_id,
                'image_path': os.path.join("train", label, f"{img_id}.png"),
                'text': text,
                'label': label.lower(),
                'split': 'train'
            })
            train_success += 1
        except:
            continue
    
    print(f"   Train: {train_success} samples")
    
    # Process test data
    print(" Processing test data...")
    test_success = 0
    
    for _, row in test_df.iterrows():
        try:
            img_id = str(row[ID_COL]).strip()
            text = str(row[TEXT_COL]) if pd.notna(row[TEXT_COL]) else ""
            label = str(row[LABEL_COL]).strip() if pd.notna(row[LABEL_COL]) else ""
            
            if label.lower() in ['happiness', 'happy']:
                label = 'Happiness'
            elif label.lower() in ['sadness', 'sad']:
                label = 'Sadness'
            else:
                continue
            
            img_path = os.path.join(IMAGES_DIR, "test", label, f"{img_id}.png")
            if not os.path.exists(img_path):
                img_path = os.path.join(IMAGES_DIR, "test", label, f"{img_id}.jpg")
                if not os.path.exists(img_path):
                    continue
            
            data.append({
                'image_id': img_id,
                'image_path': os.path.join("test", label, f"{img_id}.png"),
                'text': text,
                'label': label.lower(),
                'split': 'test'
            })
            test_success += 1
        except:
            continue
    
    print(f"   Test: {test_success} samples")
    
    if len(data) == 0:
        print(" No data found! Creating dummy data...")
        return create_dummy_data()
    
    df = pd.DataFrame(data)
    
    # Split train/val
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    
    train_df, val_df = train_test_split(
        train_df, 
        test_size=0.15, 
        random_state=42, 
        stratify=train_df['label']
    )
    
    print(f"\n Final split:")
    print(f"   Training: {len(train_df)}")
    print(f"   Validation: {len(val_df)}")
    print(f"   Test: {len(test_df)}")
    
    return train_df, val_df, test_df

def create_dummy_data():
    """Create dummy data for testing when real data is not found."""
    import random
    data = []
    labels = ['happiness', 'sadness']
    
    for split, n in [('train', 500), ('val', 100), ('test', 100)]:
        for i in range(n):
            label = random.choice(labels)
            data.append({
                'image_id': f'dummy_{i}',
                'image_path': f'{split}/{label}/dummy_{i}.png',
                'text': f"Dummy text for {label}",
                'label': label,
                'split': split
            })
    
    df = pd.DataFrame(data)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df = df[df['split'] == 'val'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    
    print(f" Using DUMMY data")
    return train_df, val_df, test_df

print(" Data loading functions ready")

 Data loading functions ready


In [6]:
# DATASET CLASS

class KIDODataset(Dataset):
    """KIDO Dataset for emotion classification."""
    
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = os.path.join(IMAGES_DIR, row['image_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='white')
        
        if self.transform:
            image = self.transform(image)
        
        # Label: 0 = happiness, 1 = sadness
        label = 0 if row['label'] == 'happiness' else 1
        
        # Text
        text = row['text'] if pd.notna(row['text']) else ""
        
        return {
            'image': image,
            'text': text,
            'label': torch.tensor(label, dtype=torch.long)
        }

def create_dataloaders(train_df, val_df, test_df, batch_size=16):
    """Create DataLoaders for training, validation, and test."""
    
    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = KIDODataset(train_df, train_transform)
    val_dataset = KIDODataset(val_df, val_transform)
    test_dataset = KIDODataset(test_df, val_transform)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f" DataLoaders created (batch_size={batch_size})")
    
    return train_loader, val_loader, test_loader

print(" Dataset class ready")

 Dataset class ready


In [7]:
# TEXT ENCODERS


class BiLSTMTextEncoder(nn.Module):
    """Bi-LSTM Text Encoder with Attention."""
    
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, 2, bidirectional=True, 
                           batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context

class GRUTextEncoder(nn.Module):
    """GRU Text Encoder with Attention."""
    
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden, 2, bidirectional=True, 
                         batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        gru_out, _ = self.gru(embedded)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        context = torch.sum(attn_weights * gru_out, dim=1)
        return context

print(" Text encoders ready")

 Text encoders ready


In [8]:
# VISION & MULTIMODAL MODEL


def get_vision_encoder(name, pretrained=True):
    """Get vision backbone with feature dimension."""
    
    backbones = {
        'mobilenet_v2': (models.mobilenet_v2, 1280),
        'efficientnet_b0': (models.efficientnet_b0, 1280),
        'shufflenet_v2_x1_0': (models.shufflenet_v2_x1_0, 1024),
        'squeezenet1_1': (models.squeezenet1_1, 512),
    }
    
    if name not in backbones:
        available = list(backbones.keys())
        raise ValueError(f"Unknown model: {name}. Available: {available}")
    
    model_fn, dim = backbones[name]
    model = model_fn(pretrained=pretrained)
    
    # Remove classification head
    if hasattr(model, 'classifier'):
        model.classifier = nn.Identity()
    elif hasattr(model, 'fc'):
        model.fc = nn.Identity()
    
    # Freeze early layers
    for param in list(model.parameters())[:10]:
        param.requires_grad = False
    
    return model, dim

class MultimodalModel(nn.Module):
    """Multimodal emotion classifier combining vision and text."""
    
    def __init__(self, vision_name, text_encoder, dropout=0.5):
        super().__init__()
        
        # Vision branch
        self.vision, vdim = get_vision_encoder(vision_name)
        self.vision_proj = nn.Sequential(
            nn.Linear(vdim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Text branch
        self.text_encoder = text_encoder
        self.text_proj = nn.Sequential(
            nn.Linear(text_encoder.output_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classifier
        fusion_dim = 512 + 256
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )
        
    def forward(self, images, texts):
        # Vision
        v = self.vision(images)
        if v.dim() > 2:
            v = v.mean([2, 3])
        v = self.vision_proj(v)
        
        # Text
        t = self.text_encoder(texts)
        t = self.text_proj(t)
        
        # Fusion
        fused = torch.cat([v, t], dim=1)
        return self.classifier(fused)
    
    def get_parameters(self):
        """Return all trainable parameters."""
        return self.parameters()

print(" Multimodal model ready")

 Multimodal model ready


In [9]:
#  VOCABULARY & COLLATION


def build_vocabulary(texts, max_vocab=10000):
    """Build vocabulary from texts."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(preprocess_text(text).split())
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in word_counts.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    
    return vocab

def text_to_sequence(text, vocab, max_len):
    """Convert text to token sequence."""
    tokens = preprocess_text(text).split()[:max_len]
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    seq += [0] * (max_len - len(seq))
    return torch.tensor(seq, dtype=torch.long)

def make_collate_fn(vocab, max_len):
    """Create collate function for DataLoader."""
    def collate_fn(batch):
        images = torch.stack([item['image'] for item in batch])
        labels = torch.stack([item['label'] for item in batch])
        texts = torch.stack([
            text_to_sequence(item['text'], vocab, max_len) for item in batch
        ])
        return {'image': images, 'text': texts, 'label': labels}
    return collate_fn

print(" Vocabulary utilities ready")

 Vocabulary utilities ready


In [10]:
#  TRAINING FUNCTIONS

def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch in tqdm(loader, desc='Training', leave=False):
        images = batch['image'].to(device)
        texts = batch['text'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images, texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation', leave=False):
            images = batch['image'].to(device)
            texts = batch['text'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(images, texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    
    return total_loss / len(loader), correct / total

def train_model(model, train_loader, val_loader, model_name, 
                epochs=15, patience=3):
    """Train model with early stopping."""
    
    device = DEVICE
    model.to(device)
    
    optimizer = optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0
    best_epoch = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    

    print(f" TRAINING: {model_name}")
    print(f"   Device: {device}")
    print(f"   Epochs: {epochs}")
    print(f"   Patience: {patience}")

    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            print(f"    New best: {best_acc:.4f}")
        else:
            patience_counter += 1
            print(f"    No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"    Early stopping triggered!")
                break
    
    print(f"\n {model_name} COMPLETE!")
    print(f"   Best Accuracy: {best_acc:.4f} at epoch {best_epoch}")
    print(f"   Total epochs: {epoch+1}")
    
    return model, history, best_acc

print(" Training functions ready")

 Training functions ready


In [11]:
# SAVE FUNCTIONS


def save_model(model, optimizer, history, model_name, best_acc):
    """Save trained model to disk."""
    
    save_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")
    
    checkpoint = {
        'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_accuracy': best_acc,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'device': str(DEVICE)
    }
    
    torch.save(checkpoint, save_path)
    
    print(f" Model saved: {save_path}")
    print(f"   Best Accuracy: {best_acc:.4f}")
    print(f"   File Size: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")
    
    return save_path

def model_exists(model_name):
    """Check if model is already saved."""
    path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {model_name} already exists ({size:.1f} MB)")
        return True
    return False

print(" Save functions ready")

 Save functions ready


In [13]:
# MODEL CONFIGURATIONS


MODEL_CONFIGS = [
    {
        'name': 'MM_MobileNetV2_BiLSTM',
        'vision': 'mobilenet_v2',
        'text_encoder': 'bilstm',
        'text_hidden': 128,
        'desc': 'MobileNetV2 + Bi-LSTM'
    },
    {
        'name': 'MM_EfficientNet_B0_BiLSTM',
        'vision': 'efficientnet_b0',
        'text_encoder': 'bilstm',
        'text_hidden': 128,
        'desc': 'EfficientNet-B0 + Bi-LSTM'
    },
    {
        'name': 'MM_ShuffleNetV2_GRU',
        'vision': 'shufflenet_v2_x1_0',
        'text_encoder': 'gru',
        'text_hidden': 128,
        'desc': 'ShuffleNetV2 + GRU'
    },
    {
        'name': 'MM_SqueezeNet_BiLSTM',
        'vision': 'squeezenet1_1',
        'text_encoder': 'bilstm',
        'text_hidden': 64,
        'desc': 'SqueezeNet + Bi-LSTM (small)'
    },
    {
        'name': 'MM_MobileNetV2_GRU',
        'vision': 'mobilenet_v2',
        'text_encoder': 'gru',
        'text_hidden': 128,
        'desc': 'MobileNetV2 + GRU'
    },
    {
        'name': 'MM_EfficientNet_B0_GRU',
        'vision': 'efficientnet_b0',
        'text_encoder': 'gru',
        'text_hidden': 128,
        'desc': 'EfficientNet-B0 + GRU'
    },
    {
        'name': 'MM_ShuffleNetV2_BiLSTM',
        'vision': 'shufflenet_v2_x1_0',
        'text_encoder': 'bilstm',
        'text_hidden': 128,
        'desc': 'ShuffleNetV2 + Bi-LSTM'
    },
    {
        'name': 'MM_SqueezeNet_GRU',
        'vision': 'squeezenet1_1',
        'text_encoder': 'gru',
        'text_hidden': 64,
        'desc': 'SqueezeNet + GRU (small)'
    },
]

def create_text_encoder(config, vocab_size):
    """Create text encoder based on config."""
    if config['text_encoder'] == 'bilstm':
        return BiLSTMTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    elif config['text_encoder'] == 'gru':
        return GRUTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    else:
        raise ValueError(f"Unknown text encoder: {config['text_encoder']}")

print(" Models to train:")
for i, cfg in enumerate(MODEL_CONFIGS, 1):
    print(f"   {i}. {cfg['name']:35} | {cfg['desc']}")
print(f"Total: {len(MODEL_CONFIGS)} models")

 Models to train:
   1. MM_MobileNetV2_BiLSTM               | MobileNetV2 + Bi-LSTM
   2. MM_EfficientNet_B0_BiLSTM           | EfficientNet-B0 + Bi-LSTM
   3. MM_ShuffleNetV2_GRU                 | ShuffleNetV2 + GRU
   4. MM_SqueezeNet_BiLSTM                | SqueezeNet + Bi-LSTM (small)
   5. MM_MobileNetV2_GRU                  | MobileNetV2 + GRU
   6. MM_EfficientNet_B0_GRU              | EfficientNet-B0 + GRU
   7. MM_ShuffleNetV2_BiLSTM              | ShuffleNetV2 + Bi-LSTM
   8. MM_SqueezeNet_GRU                   | SqueezeNet + GRU (small)
Total: 8 models


In [15]:
# MAIN TRAINING PIPELINE

def train_all_models():
    """Train all models sequentially."""
    
    results = []
    
    # Load data
    print(" LOADING DATA")
    train_df, val_df, test_df = load_kido_data()
    
    # Build vocabulary
    print(" BUILDING VOCABULARY")  
    all_texts = train_df['text'].tolist() + val_df['text'].tolist()
    vocab = build_vocabulary(all_texts)
    vocab_size = len(vocab)
    print(f" Vocabulary size: {vocab_size}")
    
    # Create dataloaders
    print(" CREATING DATALOADERS")

    train_loader, val_loader, test_loader = create_dataloaders(
        train_df, val_df, test_df, BATCH_SIZE
    )
    
    # Create collate function
    collate_fn = make_collate_fn(vocab, MAX_TEXT_LENGTH)
    
    # Train each model
    print("STARTING TRAINING")

    
    for config in MODEL_CONFIGS:
        model_name = config['name']        
        print(f"# {model_name}")
        print(f"# {config['desc']}")
        
        # Check if already saved
        if model_exists(model_name):
            results.append({
                'model_name': model_name,
                'status': 'skipped'
            })
            continue
        
        try:
            # Create text encoder
            text_enc = create_text_encoder(config, vocab_size)
            
            # Create model
            model = MultimodalModel(config['vision'], text_enc)
            
            total_params = sum(p.numel() for p in model.parameters()) / 1e6
            print(f"✓ Parameters: {total_params:.2f}M")
            
            # Create custom dataloaders
            train_loader_custom = DataLoader(
                train_loader.dataset,
                batch_size=BATCH_SIZE,
                shuffle=True,
                collate_fn=collate_fn
            )
            val_loader_custom = DataLoader(
                val_loader.dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                collate_fn=collate_fn
            )
            
            # Train
            trained_model, history, best_acc = train_model(
                model=model,
                train_loader=train_loader_custom,
                val_loader=val_loader_custom,
                model_name=model_name,
                epochs=EPOCHS,
                patience=EARLY_STOPPING_PATIENCE
            )
            
            # Save
            save_model(
                model=trained_model,
                optimizer=optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
                history=history,
                model_name=model_name,
                best_acc=best_acc
            )
            
            results.append({
                'model_name': model_name,
                'best_accuracy': best_acc,
                'status': 'completed'
            })
            
            # Clear GPU memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f" Error: {e}")
            results.append({
                'model_name': model_name,
                'status': 'failed',
                'error': str(e)
            })
    
    # Save summary
    print(" SAVING SUMMARY")
    df = pd.DataFrame(results)
    df.to_csv(os.path.join(MODEL_DIR, 'training_summary.csv'), index=False)
    print(f"✓ Summary saved: {MODEL_DIR}/training_summary.csv")
    
    return results

print(" Training pipeline ready!")

 Training pipeline ready!


In [16]:
# RUN TRAINING

print(" STARTING TRAINING FOR ALL MODELS")
print(f"   Total Models: {len(MODEL_CONFIGS)}")
print(f"   Epochs: {EPOCHS}")
print(f"   Patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Device: {DEVICE}")


results = train_all_models()


print(" FINAL SUMMARY")


completed = 0
for r in results:
    if r['status'] == 'completed':
        print(f" {r['model_name']:35} | {r['best_accuracy']:.4f}")
        completed += 1
    elif r['status'] == 'skipped':
        print(f"  {r['model_name']:35} | Already saved")
    else:
        print(f" {r['model_name']:35} | {r.get('error', 'Failed')}")

print(f" Completed: {completed}/{len(MODEL_CONFIGS)} models")
print(f" Models saved to: {MODEL_DIR}")

 STARTING TRAINING FOR ALL MODELS
   Total Models: 8
   Epochs: 15
   Patience: 3
   Batch Size: 16
   Device: cpu
 LOADING DATA
 Loading CSV files...
   Train CSV: 9228 rows
   Test CSV: 1632 rows

 Processing training data...
   Train: 9228 samples
 Processing test data...
   Test: 1632 samples

 Final split:
   Training: 7843
   Validation: 1385
   Test: 1632
 BUILDING VOCABULARY
 Vocabulary size: 3423
 CREATING DATALOADERS
 DataLoaders created (batch_size=16)
STARTING TRAINING
# MM_MobileNetV2_BiLSTM
# MobileNetV2 + Bi-LSTM


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:05<00:00, 2.70MB/s]


✓ Parameters: 5.01M
 TRAINING: MM_MobileNetV2_BiLSTM
   Device: cpu
   Epochs: 15
   Patience: 3


Epoch 1/15 | Train Loss: 0.2881 | Val Loss: 0.1532 | Val Acc: 0.9379
    New best: 0.9379


Epoch 2/15 | Train Loss: 0.1772 | Val Loss: 0.1394 | Val Acc: 0.9466
    New best: 0.9466


...

Epoch 10/15 | Train Loss: 0.0556 | Val Loss: 0.1680 | Val Acc: 0.9567
    No improvement (1/3)


Epoch 11/15 | Train Loss: 0.0393 | Val Loss: 0.1706 | Val Acc: 0.9567
    No improvement (2/3)


Epoch 12/15 | Train Loss: 0.0396 | Val Loss: 0.1690 | Val Acc: 0.9588
    No improvement (3/3)
    Early stopping triggered!

 MM_EfficientNet_B0_GRU COMPLETE!
   Best Accuracy: 0.9596 at epoch 9
   Total epochs: 12
 Model saved: Emotion_Models/MM_EfficientNet_B0_GRU_final.pt
   Best Accuracy: 0.9596
   File Size: 25.42 MB
# MM_ShuffleNetV2_BiLSTM
# ShuffleNetV2 + Bi-LSTM


Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\shufflenetv2_x1-5666bf0f80.pth
Downloading: "https://download.pytorch.org/models/squeezenet1_1-b8a52dc0.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\squeezenet1_1-b8a52dc0.pth


 Error: <urlopen error [Errno 11001] getaddrinfo failed>
# MM_SqueezeNet_GRU
# SqueezeNet + GRU (small)
 Error: <urlopen error [Errno 11001] getaddrinfo failed>
 SAVING SUMMARY
✓ Summary saved: Emotion_Models//training_summary.csv
 FINAL SUMMARY
 MM_MobileNetV2_BiLSTM               | 0.9632
 MM_EfficientNet_B0_BiLSTM           | 0.9588
 MM_ShuffleNetV2_GRU                 | <urlopen error [Errno 11001] getaddrinfo failed>
 MM_SqueezeNet_BiLSTM                | <urlopen error [Errno 11001] getaddrinfo failed>
 MM_MobileNetV2_GRU                  | 0.9617
 MM_EfficientNet_B0_GRU              | 0.9596
 MM_ShuffleNetV2_BiLSTM              | <urlopen error [Errno 11001] getaddrinfo failed>
 MM_SqueezeNet_GRU                   | <urlopen error [Errno 11001] getaddrinfo failed>
 Completed: 4/8 models
 Models saved to: Emotion_Models/


In [23]:
# PRE-DOWNLOAD ALL REQUIRED WEIGHTS 

import torchvision.models as models
import socket
import time

# Set longer timeout
socket.setdefaulttimeout(60)

print(" DOWNLOADING PRETRAINED WEIGHTS FOR FAILED MODELS")

# Models that failed
required_models = [
    'shufflenet_v2_x1_0',
    'squeezenet1_1'
]

for name in required_models:
    try:
        print(f"   Downloading {name}...", end=" ")
        model_fn = getattr(models, name)
        model_fn(pretrained=True)
        print(" Done")
    except Exception as e:
        print(f" Failed: {e}")
        print(f"   Retrying...")
        time.sleep(3)
        try:
            model_fn = getattr(models, name)
            model_fn(pretrained=True)
            print(f"   {name} downloaded on retry")
        except:
            print(f"   {name} will use random weights (pretrained=False)")

print(" Pre-download complete!")

 DOWNLOADING PRETRAINED WEIGHTS FOR FAILED MODELS
 Pre-download complete!


In [24]:
#TRAIN FAILED MODELS ONLY


def train_failed_models():
    """Train only the models that failed previously."""
    
    results = []
    
    
    for config in FAILED_MODELS_CONFIG:
        model_name = config['name']
        
        print(f"# {model_name}")
        print(f"# {config['desc']}")

        
        # Check if model already exists (maybe user manually fixed it)
        model_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")
        if os.path.exists(model_path) and os.path.getsize(model_path) > 0.1:
            size = os.path.getsize(model_path) / 1024 / 1024
            print(f"  Model already exists: {model_path} ({size:.2f} MB)")
            print(f"    Skipping (already trained)")
            results.append({
                'model_name': model_name,
                'status': 'already_exists',
                'size_mb': size
            })
            continue
        
        try:
            # Create text encoder
            text_enc = create_text_encoder(config, vocab_size)
            
            # Create model
            model = MultimodalModel(config['vision'], text_enc)
            
            total_params = sum(p.numel() for p in model.parameters()) / 1e6
            print(f"✓ Parameters: {total_params:.2f}M")
            
            # Create custom dataloaders
            train_loader_custom = DataLoader(
                train_loader.dataset,
                batch_size=BATCH_SIZE,
                shuffle=True,
                collate_fn=collate_fn
            )
            val_loader_custom = DataLoader(
                val_loader.dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                collate_fn=collate_fn
            )
            
            # Train
            trained_model, history, best_acc = train_model(
                model=model,
                train_loader=train_loader_custom,
                val_loader=val_loader_custom,
                model_name=model_name,
                epochs=EPOCHS,
                patience=EARLY_STOPPING_PATIENCE
            )
            
            # Save
            optimizer = optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            save_model(
                model=trained_model,
                optimizer=optimizer,
                history=history,
                model_name=model_name,
                best_acc=best_acc
            )
            
            results.append({
                'model_name': model_name,
                'best_accuracy': best_acc,
                'status': 'completed'
            })
            
            # Clear GPU memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f" Error training {model_name}: {e}")
            results.append({
                'model_name': model_name,
                'status': 'failed',
                'error': str(e)
            })
    
    # Save summary
    print(" SAVING RETRAINING SUMMARY")
    df = pd.DataFrame(results)
    df.to_csv(os.path.join(MODEL_DIR, 'retraining_summary.csv'), index=False)
    print(f"✓ Summary saved: {MODEL_DIR}/retraining_summary.csv")
    
    return results

print(" Retraining pipeline ready!")

 Retraining pipeline ready!


In [25]:
# LOAD DATA 

def preprocess_text(text):
    """Clean and preprocess text."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_kido_data():
    """Load KIDO dataset from local structure."""
    data = []
    
    if not os.path.exists(TRAIN_CSV) or not os.path.exists(TEST_CSV):
        print("❌ CSV files not found!")
        return create_dummy_data()
    
    print(" Loading CSV files...")
    train_df = pd.read_csv(TRAIN_CSV, header=None)
    test_df = pd.read_csv(TEST_CSV, header=None)
    
    print(f"   Train CSV: {len(train_df)} rows")
    print(f"   Test CSV: {len(test_df)} rows")
    
    ID_COL, TEXT_COL, LABEL_COL = 0, 2, 3
    
    print(" Processing data...")
    for split, df in [('train', train_df), ('test', test_df)]:
        success = 0
        for _, row in df.iterrows():
            try:
                img_id = str(row[ID_COL]).strip()
                text = str(row[TEXT_COL]) if pd.notna(row[TEXT_COL]) else ""
                label = str(row[LABEL_COL]).strip() if pd.notna(row[LABEL_COL]) else ""
                
                if label.lower() in ['happiness', 'happy']:
                    label = 'Happiness'
                elif label.lower() in ['sadness', 'sad']:
                    label = 'Sadness'
                else:
                    continue
                
                img_path = os.path.join(IMAGES_DIR, split, label, f"{img_id}.png")
                if not os.path.exists(img_path):
                    img_path = os.path.join(IMAGES_DIR, split, label, f"{img_id}.jpg")
                    if not os.path.exists(img_path):
                        continue
                
                data.append({
                    'image_id': img_id,
                    'image_path': os.path.join(split, label, f"{img_id}.png"),
                    'text': text,
                    'label': label.lower(),
                    'split': split
                })
                success += 1
            except:
                continue
        print(f"   {split}: {success} samples")
    
    if len(data) == 0:
        return create_dummy_data()
    
    df = pd.DataFrame(data)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    
    train_df, val_df = train_test_split(
        train_df, 
        test_size=0.15, 
        random_state=42, 
        stratify=train_df['label']
    )
    
    print(f"\n Final split:")
    print(f"   Training: {len(train_df)}")
    print(f"   Validation: {len(val_df)}")
    print(f"   Test: {len(test_df)}")
    
    return train_df, val_df, test_df

def create_dummy_data():
    """Create dummy data for testing."""
    import random
    data = []
    labels = ['happiness', 'sadness']
    for split, n in [('train', 500), ('val', 100), ('test', 100)]:
        for i in range(n):
            label = random.choice(labels)
            data.append({
                'image_id': f'dummy_{i}',
                'image_path': f'{split}/{label}/dummy_{i}.png',
                'text': f"Dummy text for {label}",
                'label': label,
                'split': split
            })
    df = pd.DataFrame(data)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df = df[df['split'] == 'val'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    print(f" Using DUMMY data")
    return train_df, val_df, test_df

# Load data
print("\n LOADING DATA...")
train_df, val_df, test_df = load_kido_data()


 LOADING DATA...
 Loading CSV files...
   Train CSV: 9228 rows
   Test CSV: 1632 rows
 Processing data...
   train: 9228 samples
   test: 1632 samples

 Final split:
   Training: 7843
   Validation: 1385
   Test: 1632


In [27]:
# BUILD VOCABULARY & CREATE DATALOADERS

def build_vocabulary(texts, max_vocab=10000):
    """Build vocabulary from texts."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(preprocess_text(text).split())
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in word_counts.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    
    return vocab

def text_to_sequence(text, vocab, max_len):
    """Convert text to token sequence."""
    tokens = preprocess_text(text).split()[:max_len]
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    seq += [0] * (max_len - len(seq))
    return torch.tensor(seq, dtype=torch.long)

def make_collate_fn(vocab, max_len):
    """Create collate function for DataLoader."""
    def collate_fn(batch):
        images = torch.stack([item['image'] for item in batch])
        labels = torch.stack([item['label'] for item in batch])
        texts = torch.stack([
            text_to_sequence(item['text'], vocab, max_len) for item in batch
        ])
        return {'image': images, 'text': texts, 'label': labels}
    return collate_fn

class KIDODataset(Dataset):
    """KIDO Dataset for emotion classification."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(IMAGES_DIR, row['image_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='white')
        
        if self.transform:
            image = self.transform(image)
        
        label = 0 if row['label'] == 'happiness' else 1
        text = row['text'] if pd.notna(row['text']) else ""
        
        return {
            'image': image,
            'text': text,
            'label': torch.tensor(label, dtype=torch.long)
        }

def create_dataloaders(train_df, val_df, test_df, batch_size=16):
    """Create DataLoaders."""
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = KIDODataset(train_df, train_transform)
    val_dataset = KIDODataset(val_df, val_transform)
    test_dataset = KIDODataset(test_df, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f" DataLoaders created (batch_size={batch_size})")
    return train_loader, val_loader, test_loader

# Build vocabulary
print("\n BUILDING VOCABULARY...")
all_texts = train_df['text'].tolist() + val_df['text'].tolist()
vocab = build_vocabulary(all_texts)
vocab_size = len(vocab)
print(f" Vocabulary size: {vocab_size}")

# Create dataloaders
print("\n CREATING DATALOADERS...")
train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df, BATCH_SIZE
)

collate_fn = make_collate_fn(vocab, MAX_TEXT_LENGTH)


 BUILDING VOCABULARY...
 Vocabulary size: 3423

 CREATING DATALOADERS...
 DataLoaders created (batch_size=16)


In [28]:
# MODEL COMPONENTS 

class BiLSTMTextEncoder(nn.Module):
    """Bi-LSTM Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, 2, bidirectional=True, 
                           batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context

class GRUTextEncoder(nn.Module):
    """GRU Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden, 2, bidirectional=True, 
                         batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        gru_out, _ = self.gru(embedded)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        context = torch.sum(attn_weights * gru_out, dim=1)
        return context

def get_vision_encoder(name, pretrained=True):
    """Get vision backbone with feature dimension."""
    backbones = {
        'mobilenet_v2': (models.mobilenet_v2, 1280),
        'efficientnet_b0': (models.efficientnet_b0, 1280),
        'shufflenet_v2_x1_0': (models.shufflenet_v2_x1_0, 1024),
        'squeezenet1_1': (models.squeezenet1_1, 512),
    }
    
    if name not in backbones:
        available = list(backbones.keys())
        raise ValueError(f"Unknown model: {name}. Available: {available}")
    
    model_fn, dim = backbones[name]
    model = model_fn(pretrained=pretrained)
    
    if hasattr(model, 'classifier'):
        model.classifier = nn.Identity()
    elif hasattr(model, 'fc'):
        model.fc = nn.Identity()
    
    for param in list(model.parameters())[:10]:
        param.requires_grad = False
    
    return model, dim

class MultimodalModel(nn.Module):
    """Multimodal emotion classifier."""
    def __init__(self, vision_name, text_encoder, dropout=0.5):
        super().__init__()
        
        self.vision, vdim = get_vision_encoder(vision_name)
        self.vision_proj = nn.Sequential(
            nn.Linear(vdim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.text_encoder = text_encoder
        self.text_proj = nn.Sequential(
            nn.Linear(text_encoder.output_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        fusion_dim = 512 + 256
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )
        
    def forward(self, images, texts):
        v = self.vision(images)
        if v.dim() > 2:
            v = v.mean([2, 3])
        v = self.vision_proj(v)
        
        t = self.text_encoder(texts)
        t = self.text_proj(t)
        
        fused = torch.cat([v, t], dim=1)
        return self.classifier(fused)
    
    def get_parameters(self):
        return self.parameters()

def create_text_encoder(config, vocab_size):
    """Create text encoder based on config."""
    if config['text_encoder'] == 'bilstm':
        return BiLSTMTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    elif config['text_encoder'] == 'gru':
        return GRUTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    else:
        raise ValueError(f"Unknown text encoder: {config['text_encoder']}")

print("Model components ready")

Model components ready


In [29]:
# FAILED MODELS CONFIGURATION (ONLY THE ONES THAT FAILED)

# Only the models that failed due to network errors
FAILED_MODELS_CONFIG = [
    {
        'name': 'MM_ShuffleNetV2_GRU',
        'vision': 'shufflenet_v2_x1_0',
        'text_encoder': 'gru',
        'text_hidden': 128,
        'desc': 'ShuffleNetV2 + GRU'
    },
    {
        'name': 'MM_SqueezeNet_BiLSTM',
        'vision': 'squeezenet1_1',
        'text_encoder': 'bilstm',
        'text_hidden': 64,
        'desc': 'SqueezeNet + Bi-LSTM (small)'
    },
    {
        'name': 'MM_ShuffleNetV2_BiLSTM',
        'vision': 'shufflenet_v2_x1_0',
        'text_encoder': 'bilstm',
        'text_hidden': 128,
        'desc': 'ShuffleNetV2 + Bi-LSTM'
    },
    {
        'name': 'MM_SqueezeNet_GRU',
        'vision': 'squeezenet1_1',
        'text_encoder': 'gru',
        'text_hidden': 64,
        'desc': 'SqueezeNet + GRU (small)'
    },
]

print(" FAILED MODELS TO RETRAIN:")
for i, cfg in enumerate(FAILED_MODELS_CONFIG, 1):
    print(f"   {i}. {cfg['name']:35} | {cfg['desc']}")
print(f"Total: {len(FAILED_MODELS_CONFIG)} models")
print("\n These models will be retrained with pretrained weights")

 FAILED MODELS TO RETRAIN:
   1. MM_ShuffleNetV2_GRU                 | ShuffleNetV2 + GRU
   2. MM_SqueezeNet_BiLSTM                | SqueezeNet + Bi-LSTM (small)
   3. MM_ShuffleNetV2_BiLSTM              | ShuffleNetV2 + Bi-LSTM
   4. MM_SqueezeNet_GRU                   | SqueezeNet + GRU (small)
Total: 4 models

 These models will be retrained with pretrained weights


In [30]:
# TRAIN FAILED MODELS ONLY

print(" RETRAINING FAILED MODELS ONLY")
print(f"   Models to train: {len(FAILED_MODELS_CONFIG)}")
print(f"   Epochs: {EPOCHS}")
print(f"   Patience: {EARLY_STOPPING_PATIENCE}")


results = []

for config in FAILED_MODELS_CONFIG:
    model_name = config['name']

    print(f"# {model_name}")
    print(f"# {config['desc']}")
    
    # Check if already saved (in case it was trained successfully this time)
    if model_exists(model_name):
        results.append({
            'model_name': model_name,
            'status': 'skipped',
            'best_accuracy': None
        })
        continue
    
    try:
        # Create text encoder
        text_enc = create_text_encoder(config, vocab_size)
        
        # Create model with pretrained=True (now should work after pre-download)
        print(f" Creating model with pretrained weights...")
        model = MultimodalModel(config['vision'], text_enc)
        
        total_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f" Parameters: {total_params:.2f}M")
        
        # Create custom dataloaders
        train_loader_custom = DataLoader(
            train_loader.dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_fn
        )
        val_loader_custom = DataLoader(
            val_loader.dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn
        )
        
        # Train
        trained_model, history, best_acc = train_model(
            model=model,
            train_loader=train_loader_custom,
            val_loader=val_loader_custom,
            model_name=model_name,
            epochs=EPOCHS,
            patience=EARLY_STOPPING_PATIENCE
        )
        
        # Save immediately
        save_model(
            model=trained_model,
            optimizer=optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
            history=history,
            model_name=model_name,
            best_acc=best_acc
        )
        
        results.append({
            'model_name': model_name,
            'best_accuracy': best_acc,
            'status': 'completed'
        })
        
        # Clear GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"Error: {e}")
        results.append({
            'model_name': model_name,
            'status': 'failed',
            'error': str(e),
            'best_accuracy': None
        })

# Save summary
print(" SAVING SUMMARY")
df = pd.DataFrame(results)
summary_path = os.path.join(MODEL_DIR, 'continuation_summary.csv')
df.to_csv(summary_path, index=False)
print(f" Summary saved: {summary_path}")

 RETRAINING FAILED MODELS ONLY
   Models to train: 4
   Epochs: 15
   Patience: 3
# MM_ShuffleNetV2_GRU
# ShuffleNetV2 + GRU
 Creating model with pretrained weights...
 Parameters: 3.70M
 TRAINING: MM_ShuffleNetV2_GRU
   Device: cpu
   Epochs: 15
   Patience: 3


Epoch 1/15 | Train Loss: 0.2804 | Val Loss: 0.1734 | Val Acc: 0.9105
    New best: 0.9105


Epoch 2/15 | Train Loss: 0.1715 | Val Loss: 0.1292 | Val Acc: 0.9495
    New best: 0.9495


Epoch 3/15 | Train Loss: 0.1415 | Val Loss: 0.1300 | Val Acc: 0.9401
    No improvement (1/3)


Epoch 4/15 | Train Loss: 0.1242 | Val Loss: 0.1736 | Val Acc: 0.9394
    No improvement (2/3)


...

Epoch 5/15 | Train Loss: 0.0986 | Val Loss: 0.1263 | Val Acc: 0.9545
    New best: 0.9545


Epoch 6/15 | Train Loss: 0.0820 | Val Loss: 0.1397 | Val Acc: 0.9516
    No improvement (1/3)


Epoch 7/15 | Train Loss: 0.0766 | Val Loss: 0.1536 | Val Acc: 0.9458
    No improvement (2/3)


Epoch 8/15 | Train Loss: 0.0631 | Val Loss: 0.1322 | Val Acc: 0.9531
    No improvement (3/3)
    Early stopping triggered!

 MM_ShuffleNetV2_BiLSTM COMPLETE!
   Best Accuracy: 0.9545 at epoch 5
   Total epochs: 8
 Model saved: Emotion_Models/MM_ShuffleNetV2_BiLSTM_final.pt
   Best Accuracy: 0.9545
   File Size: 15.09 MB
# MM_SqueezeNet_GRU
# SqueezeNet + GRU (small)
 Creating model with pretrained weights...
 Parameters: 2.46M
 TRAINING: MM_SqueezeNet_GRU
   Device: cpu
   Epochs: 15
   Patience: 3


Error: mat1 and mat2 shapes cannot be multiplied (16x86528 and 512x512)
 SAVING SUMMARY
 Summary saved: Emotion_Models/continuation_summary.csv
